# Seminário 2: Geração e exploração de um banco de dados acerca dos pesquisadores do ICMC-USP

Neste notebook encontram-se todas as visualizações descritas em [nosso relatório](https://github.com/de-abreu/visualizacao_computacional/blob/main/seminario2/README.md), de tal forma que o leitor possa interatir com as mesmas. Para mais informações sobre as técnicas de visualização ou o contexto em que estas estão sendo empregadas, recomenda-se a leitura deste documento.

Após executar as células deste notebook, as visualizações aparecerão nesta página. Entretanto, recomenda-se acessar as visualizações abrindo uma nova aba para as páginas web em que estas são geradas. Os links para acessar estas encontram-se listadas abaixo:

- [Diagrama de arcos](http://127.0.0.1:8051/)
- [Gráfico de dispersão](http://127.0.0.1:8052/)
- [Gráfico de linhas](http://127.0.0.1:8053/)

## Dependências

In [1]:
from arc_diagram.collab_dashboard import create_collab_dashboard
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine("sqlite:///database/lattes.db") # Acesso ao banco de dados

## Diagrama de Arcos

In [2]:
collaborations_query = """
WITH article_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        a.title AS collaboration,
        'artigo' AS type,
        a.year AS start,
        a.year AS end
    FROM authorship au1
    JOIN authorship au2 ON au1.article_id = au2.article_id AND au1.author_id < au2.author_id
    JOIN researchers r1 ON au1.author_id = r1.lattes_id
    JOIN researchers r2 ON au2.author_id = r2.lattes_id
    JOIN articles a ON au1.article_id = a.id
),
project_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        p.name AS collaboration,
        'projeto' AS type,
        p.start AS start,
        CAST(COALESCE(p.end, strftime('%Y', 'now')) AS INTEGER) AS end
    FROM participation p1
    JOIN participation p2 ON p1.project_id = p2.project_id AND p1.participant_id < p2.participant_id
    JOIN researchers r1 ON p1.participant_id = r1.lattes_id
    JOIN researchers r2 ON p2.participant_id = r2.lattes_id
    JOIN projects p ON p1.project_id = p.id
)
SELECT * FROM article_collaborations
UNION ALL
SELECT * FROM project_collaborations
"""

# Load dataframe with data extracted from the database query
try:
    collaborations: pd.DataFrame = pd.read_sql_query(collaborations_query, engine)
except Exception as e:
    print(f"✗ Erro ao carregar dados de colaboração do banco de dados: {e}")
    print(
        "  - Verifique se o banco de dados existe e contém as tabelas necessárias"
    )
    raise

print("✓ Dados de colaboração carregados com sucesso")
print(f"  - {len(collaborations)} registros carregados")

# Create and run the Dash app for the Arc Diagram visualization
app = create_collab_dashboard(
    collab_df=collaborations,
    title="Colaborações entre Professores do ICMC, em artigos e projetos de pesquisa",
    legend_title="Colaborações",
)
app.run(host="127.0.0.1", port=8051)

✓ Dados de colaboração carregados com sucesso
  - 807 registros carregados
